# Urban Pulse NYC - Analyst Tasks 7 to 11 (status, resolution time and outliers)

**Rules used in this notebook:**
- A *valid resolution duration* is a row where `resolution_hours` is not blank (114,660 rows). Rows closed before they were created were left blank on purpose.
- Group medians are reported only when a group has **at least 100 valid durations**, because a median from a handful of records is not reliable.
- Outliers are **flagged and kept**, never deleted.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

PROJECT = "C:/Users/skmah/urban_pulse"
if os.path.isdir(PROJECT):
    os.chdir(PROJECT)
print("Working folder:", os.getcwd())

os.makedirs("figures", exist_ok=True)
os.makedirs("sql", exist_ok=True)

engine = create_engine("mysql+pymysql://root:@localhost:3306/urban_pulse_db")
pd.options.display.width = 150
pd.options.display.float_format = "{:,.2f}".format

MIN_N = 100           # minimum valid durations for a group median to be reported
QUERIES = {}

def task(num, title, sql):
    """Run a SQL query, remember it, and return the result as a DataFrame."""
    QUERIES[num] = (title, sql.strip())
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def save_fig(num, name):
    path = f"figures/task{num:02d}_{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved chart:", path)

with engine.connect() as conn:
    TOTAL = int(pd.read_sql(text("SELECT COUNT(*) AS n FROM service_requests"), conn).loc[0, "n"])
    VALID = int(pd.read_sql(text("SELECT COUNT(resolution_hours) AS n FROM service_requests"), conn).loc[0, "n"])
    print("Server version:", pd.read_sql(text("SELECT VERSION() AS v"), conn).loc[0, "v"])
    try:   # the median queries need window functions (MySQL 8+ / MariaDB 10.2+)
        pd.read_sql(text("SELECT ROW_NUMBER() OVER (ORDER BY unique_key) AS r FROM service_requests LIMIT 1"), conn)
        print("Window functions: supported")
    except Exception as e:
        print("Window functions NOT supported.")
print(f"Connected. Rows: {TOTAL:,} | valid resolution durations: {VALID:,}")

## Task 7 - Distribution of recorded request statuses
**Question:** What share of requests has each recorded status? Also: how many rows have a closed date even though the status is not "Closed"?

In [ ]:
df7 = task(7, "Distribution of recorded request statuses",
    """SELECT status,
              COUNT(*) AS requests,
              ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM service_requests), 2) AS pct_of_all,
              CAST(SUM(is_closed) AS SIGNED) AS rows_with_closed_date
       FROM service_requests
       GROUP BY status
       ORDER BY requests DESC""")
display(df7)
not_closed = df7[df7["status"] != "Closed"]
print("n =", f"{TOTAL:,}")
print("Rows NOT marked 'Closed':", f"{int(not_closed['requests'].sum()):,}",
      "(", round(not_closed["pct_of_all"].sum(), 2), "% )")
print("Of those, rows that still have a closed date:", int(not_closed["rows_with_closed_date"].sum()),
      "<- should equal the 482 status/closed mismatch flags")

plt.figure(figsize=(8, 4.5))
bars = plt.barh(df7["status"][::-1], df7["requests"][::-1], color="slateblue")
for b, p in zip(bars, df7["pct_of_all"][::-1]):
    plt.text(b.get_width(), b.get_y() + b.get_height() / 2, f"  {p}%", va="center")
plt.xlim(0, df7["requests"].max() * 1.15)
plt.xlabel("Requests (3% sample)")
plt.title("Task 7: Recorded request status")
save_fig(7, "status_distribution")
plt.show()

## Task 8 - Descriptive statistics for valid resolution durations
**Question:** What are the centre, spread and shape of `resolution_hours`? The table also compares MySQL results with your Colab Cell 11 results, as a cross-check.

In [ ]:
# Part A: SQL aggregate (what MySQL can calculate directly)
df8a = task("8a", "Resolution duration - SQL aggregates",
    """SELECT COUNT(resolution_hours)              AS valid_durations,
              ROUND(AVG(resolution_hours), 2)      AS mean_hours,
              ROUND(MIN(resolution_hours), 2)      AS min_hours,
              ROUND(MAX(resolution_hours), 2)      AS max_hours,
              ROUND(STDDEV_SAMP(resolution_hours), 2) AS std_hours
       FROM service_requests""")
display(df8a)

# Part B: pull the valid durations into pandas for median, quartiles, skewness, percentiles
res = task("8b", "Resolution duration - values for Python statistics",
    """SELECT resolution_hours
       FROM service_requests
       WHERE resolution_hours IS NOT NULL""")["resolution_hours"]

mysql_stats = {
    "count": len(res),
    "mean": res.mean(),
    "median": res.median(),
    "mode (rounded to 1h)": res.round(0).mode().iloc[0],
    "min": res.min(),
    "max": res.max(),
    "range": res.max() - res.min(),
    "Q1": res.quantile(0.25),
    "Q3": res.quantile(0.75),
    "IQR": res.quantile(0.75) - res.quantile(0.25),
    "variance": res.var(),
    "std": res.std(),
    "skewness": res.skew(),
    "kurtosis": res.kurt(),
    "p90": res.quantile(0.90),
    "p95": res.quantile(0.95),
    "p99": res.quantile(0.99),
}
colab_stats = {   # from your Colab Cell 11 screenshot
    "count": 114660, "mean": 166.85, "median": 7.66, "mode (rounded to 1h)": 1.00, "min": 0.00,
    "max": 9261.19, "range": 9261.19, "Q1": 1.17, "Q3": 77.50, "IQR": 76.33,
    "variance": 270416.05, "std": 520.02, "skewness": 6.40, "kurtosis": 56.76,
    "p90": 398.97, "p95": 910.52, "p99": 2687.17,
}
df8 = pd.DataFrame({
    "this notebook (MySQL data)": pd.Series(mysql_stats),
    "Colab Cell 11": pd.Series(colab_stats),
})
df8["match"] = [("OK" if abs(a - b) <= 0.011 else "CHECK")
                for a, b in zip(df8.iloc[:, 0], df8.iloc[:, 1])]
display(df8)
print("n (valid durations) =", f"{len(res):,}", "of", f"{TOTAL:,}", "rows")

plt.figure(figsize=(8, 4.5))
labels = ["median", "mean", "p90", "p95", "p99"]
vals = [mysql_stats[k] for k in labels]
bars = plt.bar(labels, vals, color=["seagreen", "darkorange", "steelblue", "steelblue", "steelblue"])
for b, v in zip(bars, vals):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:,.1f}", ha="center", va="bottom")
plt.ylabel("Resolution hours")
plt.title("Task 8: Median vs mean vs upper percentiles")
save_fig(8, "median_vs_mean_percentiles")
plt.show()

## Task 9 - Median resolution duration by complaint category
**Question:** Which complaint types take the longest (and shortest) to reach a recorded closure? Only categories with at least 100 valid durations are compared.

In [ ]:
df9_all = task(9, "Median resolution duration by complaint category",
    """WITH ranked AS (
         SELECT complaint_type,
                resolution_hours,
                ROW_NUMBER() OVER (PARTITION BY complaint_type ORDER BY resolution_hours) AS rn,
                COUNT(*)     OVER (PARTITION BY complaint_type)                           AS n
         FROM service_requests
         WHERE resolution_hours IS NOT NULL
       )
       SELECT complaint_type,
              MAX(n) AS valid_durations,
              ROUND(AVG(CASE WHEN rn IN (FLOOR((n + 1) / 2), CEIL((n + 1) / 2))
                             THEN resolution_hours END), 2) AS median_hours,
              ROUND(AVG(resolution_hours), 2) AS mean_hours
       FROM ranked
       GROUP BY complaint_type
       ORDER BY median_hours DESC""")

df9 = df9_all[df9_all["valid_durations"] >= MIN_N].copy()
print(f"Categories with valid durations: {len(df9_all)} | with at least {MIN_N}: {len(df9)} "
      f"| covering {int(df9['valid_durations'].sum()):,} of {VALID:,} valid durations")
print("\nLongest median resolution (top 10):")
display(df9.head(10))
print("Shortest median resolution (bottom 5):")
display(df9.tail(5).sort_values("median_hours"))

top = df9.head(10).iloc[::-1]
plt.figure(figsize=(9, 5))
plt.barh([f"{c} (n={n:,})" for c, n in zip(top["complaint_type"], top["valid_durations"])],
         top["median_hours"], color="indianred")
plt.xlabel("Median resolution hours")
plt.title(f"Task 9: Longest median resolution by complaint type (n >= {MIN_N})")
save_fig(9, "median_by_complaint")
plt.show()

## Task 10 - Median resolution duration by agency
**Question:** How do agencies compare on median recorded resolution time? Agencies with fewer than 100 valid durations are listed but flagged, not ranked.

In [ ]:
df10 = task(10, "Median resolution duration by agency",
    """WITH ranked AS (
         SELECT agency,
                resolution_hours,
                ROW_NUMBER() OVER (PARTITION BY agency ORDER BY resolution_hours) AS rn,
                COUNT(*)     OVER (PARTITION BY agency)                           AS n
         FROM service_requests
         WHERE resolution_hours IS NOT NULL
       )
       SELECT agency,
              MAX(n) AS valid_durations,
              ROUND(AVG(CASE WHEN rn IN (FLOOR((n + 1) / 2), CEIL((n + 1) / 2))
                             THEN resolution_hours END), 2) AS median_hours,
              ROUND(AVG(resolution_hours), 2) AS mean_hours
       FROM ranked
       GROUP BY agency
       ORDER BY median_hours DESC""")
df10["reliable (n>=100)"] = df10["valid_durations"] >= MIN_N
display(df10)
small = df10[~df10["reliable (n>=100)"]]["agency"].tolist()
print("n =", f"{int(df10['valid_durations'].sum()):,}", "valid durations | agencies too small to compare:", small)

rel = df10[df10["reliable (n>=100)"]].sort_values("median_hours")
plt.figure(figsize=(9, 6))
plt.barh([f"{a} (n={n:,})" for a, n in zip(rel["agency"], rel["valid_durations"])],
         rel["median_hours"], color="teal")
plt.xlabel("Median resolution hours")
plt.title(f"Task 10: Median resolution hours by agency (n >= {MIN_N})")
save_fig(10, "median_by_agency")
plt.show()

## Task 11 - IQR-based outlier investigation
**Question:** How many resolution durations are statistical outliers under the 1.5 x IQR rule, where are they concentrated, and does the stored `resolution_outlier_flag` agree with a fresh calculation? Outliers are investigation prompts, **not** deletions: long service cases can be legitimate.

In [ ]:
q1, q3 = res.quantile(0.25), res.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_out = int(((res < lower) | (res > upper)).sum())

summary11 = pd.DataFrame({
    "value": [q1, q3, iqr, lower, upper, n_out, 100 * n_out / len(res)]},
    index=["Q1", "Q3", "IQR", "lower fence", "upper fence", "outliers flagged", "outlier % of valid"])
display(summary11)

chk = task("11-check", "Stored outlier flag total (valid durations only)",
    """SELECT SUM(resolution_outlier_flag) AS stored_flag_total
       FROM service_requests
       WHERE resolution_hours IS NOT NULL""")
flag_sum = int(chk.loc[0, "stored_flag_total"])
print("Fresh calculation:", n_out, "| stored flag in MySQL:", flag_sum,
      "->", "MATCH" if n_out == flag_sum else "CHECK")
print("Lower fence is negative, so there are no low outliers (durations cannot be below 0).")
print("n =", f"{len(res):,}", "valid durations")

In [ ]:
df11a = task("11a", "Outliers by agency",
    """SELECT agency,
              COUNT(*) AS valid_durations,
              CAST(SUM(resolution_outlier_flag) AS SIGNED) AS outliers,
              ROUND(100 * SUM(resolution_outlier_flag) / COUNT(*), 1) AS outlier_pct
       FROM service_requests
       WHERE resolution_hours IS NOT NULL
       GROUP BY agency
       ORDER BY outliers DESC""")
df11a["share_of_all_outliers_pct"] = 100 * df11a["outliers"] / df11a["outliers"].sum()
print("Outliers by agency (count and rate within agency):")
display(df11a.head(8))

df11b = task("11b", "Outliers by complaint type (top 10 by count)",
    """SELECT complaint_type,
              COUNT(*) AS valid_durations,
              CAST(SUM(resolution_outlier_flag) AS SIGNED) AS outliers,
              ROUND(100 * SUM(resolution_outlier_flag) / COUNT(*), 1) AS outlier_pct
       FROM service_requests
       WHERE resolution_hours IS NOT NULL
       GROUP BY complaint_type
       ORDER BY outliers DESC
       LIMIT 10""")
print("Outliers by complaint type (top 10 by count):")
display(df11b)

# Chart: counts (left) vs rates (right) - they tell different stories
by_count = df11a.head(8).iloc[::-1]
by_rate = df11a[df11a["valid_durations"] >= MIN_N].sort_values("outlier_pct", ascending=False).head(8).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(by_count["agency"], by_count["outliers"], color="firebrick")
axes[0].set_title("Outlier COUNT (top 8 agencies)")
axes[0].set_xlabel("Outliers")
axes[1].barh(by_rate["agency"], by_rate["outlier_pct"], color="darkorange")
axes[1].set_title(f"Outlier RATE within agency (n >= {MIN_N})")
axes[1].set_xlabel("% of agency's valid durations that are outliers")
fig.suptitle("Task 11: Where do the IQR outliers sit?")
save_fig(11, "outliers_by_agency")
plt.show()

## Save all queries from this notebook
Creates `sql/03_tasks_07_11.sql` - another of your required deliverables.

In [ ]:
with open("sql/03_tasks_07_11.sql", "w", encoding="utf-8") as f:
    f.write("-- Urban Pulse NYC: analyst tasks 7-11 (run against urban_pulse_db)\n")
    f.write("-- Note: medians use window functions because MySQL has no MEDIAN().\n\n")
    for num, (title, sql) in QUERIES.items():
        f.write(f"-- Task {num}: {title}\n{sql};\n\n")
print("Saved sql/03_tasks_07_11.sql with", len(QUERIES), "queries")